# Business Case — ONG World Vision
## Identification des pays prioritaires pour l'accès à l'eau potable

**Auteur :** Mégane La Rocca  
**Formation :** Data Analyst RNCP6 — Promotion Vert 11-2025  
**Date :** 21 avril 2026  
**Durée de l'exercice :** 8h

## 1. Contexte

L'ONG **World Vision** intervient à l'échelle mondiale pour fournir un accès à l'eau potable : création de services, modernisation des infrastructures, accompagnement des politiques publiques.

Jusqu'à présent, le choix des zones d'intervention s'est fait de manière **artisanale**, sans approche data-driven. L'ONG souhaite aujourd'hui **maximiser l'impact** de ses actions en s'appuyant sur les données.

## 2. Problématique

> **Quels sont les pays prioritaires où l'ONG World Vision doit concentrer ses interventions pour maximiser son impact sur l'accès à l'eau potable ?**

## 3. Objectif du livrable

Je dois produire un dashboard interactif offrant **3 vues complémentaires** :
- **Mondiale** : panorama global et identification des zones critiques
- **Régionale** : analyse comparée des continents / régions
- **Nationale** : zoom pays pour les décisions opérationnelles

## 4. Hypothèses de travail

→ Un pays prioritaire combine plusieurs critères : **forte mortalité liée à l'eau**, **faible accès à l'eau potable**, **population importante** (impact potentiel), et **stabilité politique suffisante** (faisabilité de l'intervention).

→ Les inégalités **urbain / rural** et **hommes / femmes** peuvent révéler des leviers d'action ciblés.

→ La construction d'un **score de priorité composite** permettra de hiérarchiser les pays de manière objective et argumentable.

## 5. Démarche

1. **Collecte** : chargement des 5 datasets fournis
2. **Exploration** : audit qualité, structure, relations entre datasets
3. **Préparation** : nettoyage, jointures, création de variables dérivées
4. **Analyse** : réponse aux questions métier (mortalité, politique, urbain/rural, genre)
5. **Modélisation** : construction d'un score de priorité
6. **Visualisation** : graphiques, carte, tableau croisé
7. **Conclusion** : recommandations pour les investisseurs

## 6. Datasets utilisés

- `df_population` : population mondiale par pays (2000-2018)
- `df_basicsafewater` : accès eau potable rural/urbain par pays (2000-2017)
- `df_mortality` : taux de mortalité lié à l'eau non potable (2016)
- `df_political` : stabilité politique par pays (2000-2017)
- `df_region_country` : région d'appartenance de chaque pays

## 7. Import des bibliothèques

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print("Bibliothèques importées avec succès")

Bibliothèques importées avec succès


## 8. Chargement des données

je réalise le chargement des données en mettant directement chaque id pour chaque chargement. Je trouve cela plus fluide que réassigner la variable 'id' à chaque fois. 

In [3]:
def get_file(id: str) -> pd.DataFrame:
    """Je télécharge un fichier CSV depuis Google Drive via son ID."""
    url = f'https://drive.google.com/uc?id={id}'
    return pd.read_csv(url)

df_population = get_file('1DEOHEXtyZC-FjQOoEzVUA4ZCbt4_AYFK')
df_basicsafewater = get_file('1X03LbUPaQcoiBdxgJvDcNqGIMfHhsxSz')
df_mortality = get_file('1tUVfNHS6MEarPR9oZCayCBNw-V8-fc1n')
df_political = get_file('1cpVJg3QFHmZerPbCmjJteGQYV8iprVYE')
df_region_country = get_file('1y_jygG2YhuKJ5BoWoW4M394XSHYas0Jf')

print("Chargement des 5 datasets terminé")

Chargement des 5 datasets terminé


## 9. Vue d'ensmble des datasets

In [ ]:
Afin d'optimiser le temps d'exploration, je vais:
→ Charger les 5 DataFrames dans un dictionnaire
→ Faire l'audit en boucle sur les 5 d'un coup
→ 5x moins de code 

In [7]:
# Je charge les 5 datasets dans un dictionnaire
dfs = {
    'population': df_population,
    'basicsafewater': df_basicsafewater,
    'mortality': df_mortality,
    'political': df_political,
    'region_country': df_region_country
}

# J'audite tous les datasets d'un coup
for name, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"DATASET : {name}")
    print(f"{'='*60}")
    print(f"Shape : {df.shape}")
    print(f"Colonnes : {df.columns.tolist()}")
    print(f"Doublons : {df.duplicated().sum()}")
    print(f"NaN totaux : {df.isnull().sum().sum()}")
    print(f"\nAperçu :")
    print(df.head(3))


DATASET : population
Shape : (20914, 4)
Colonnes : ['Country', 'Granularity', 'Year', 'Population']
Doublons : 0
NaN totaux : 0

Aperçu :
       Country Granularity  Year  Population
0  Afghanistan       Total  2000   20779.953
1  Afghanistan        Male  2000   10689.508
2  Afghanistan      Female  2000   10090.449

DATASET : basicsafewater
Shape : (10476, 5)
Colonnes : ['Year', 'Country', 'Granularity', 'Population using at least basic drinking-water services (%)', 'Population using safely managed drinking-water services (%)']
Doublons : 0
NaN totaux : 8251

Aperçu :
   Year      Country Granularity  \
0  2000  Afghanistan       Rural   
1  2000  Afghanistan       Total   
2  2000  Afghanistan       Urban   

   Population using at least basic drinking-water services (%)  \
0                                           21.61913             
1                                           27.77190             
2                                           49.48745             

   Population

## Synthèse de l'audit initial

### Volumétrie et structure

→ **5 datasets chargés** avec succès, pas de doublons détectés  
→ **population** : 20 914 lignes × 4 colonnes — propre, 0 NaN  
→ **political** : 3 526 lignes × 4 colonnes — propre, 0 NaN  
→ **region_country** : 194 lignes × 2 colonnes — table de référence, 0 NaN  
→ **basicsafewater** : 10 476 lignes × 5 colonnes — **8 251 NaN à investiguer**  
→ **mortality** : 549 lignes × 5 colonnes — **366 NaN à investiguer**  

### Points d'attention identifiés

→ → La colonne **"safely managed drinking-water"** concentre probablement la majorité des NaN de `basicsafewater` — indicateur avec **trop de valeurs manquantes** pour être exploité 
→ Dans `mortality`, la colonne **`WASH deaths`** a des NaN quand `Granularity` ≠ Total — à vérifier  
→ Les datasets ont tous une colonne **`Country`** et souvent **`Granularity`** → jointures et filtrages possibles  
→ `region_country` utilise des noms de colonnes différents (`COUNTRY (DISPLAY)`) → harmonisation nécessaire avant jointure  

### Prochaine étape

→ **Zoom ciblé** sur `basicsafewater` et `mortality` pour identifier précisément les colonnes exploitables  
→ Décision à prendre : **conserver ou abandonner** la colonne "safely managed drinking-water"

## 10. Exploration complémentaire: types, sats, modalités

Ce que je veux savoir maintenant pour pouvoir prendre une décision métier:
→ Types de données (certaines colonnes doivent être converties)
→ Stats descriptives des variables numériques (ordre de grandeur, outliers)
→ Zoom sur les NaN des 2 datasets à problèmes

In [8]:
for name, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"DATASET : {name}")
    print(f"{'='*60}")
    print("\n--- INFO ---")
    df.info()
    print("\n--- DESCRIBE ---")
    print(df.describe(include='all'))


DATASET : population

--- INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 20914 entries, 0 to 20913
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      20914 non-null  str    
 1   Granularity  20914 non-null  str    
 2   Year         20914 non-null  int64  
 3   Population   20914 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 973.9 KB

--- DESCRIBE ---
            Country Granularity          Year    Population
count         20914       20914  20914.000000  2.091400e+04
unique          239           5           NaN           NaN
top     Afghanistan       Total           NaN           NaN
freq             95        4430           NaN           NaN
mean            NaN         NaN   2009.046715  2.253164e+04
std             NaN         NaN      5.479195  1.000169e+05
min             NaN         NaN   2000.000000  0.000000e+00
25%             NaN         NaN   2004.000000  3.483462e+

In [9]:
# Maintenant que j'ai plus de détail, je calcule le % de remplissage des colonnes potentiellement problématiques

print("=== basicsafewater ===")
for col in df_basicsafewater.columns:
    taux = df_basicsafewater[col].notnull().mean() * 100
    print(f"{col} : {taux:.1f}% rempli")

print("\n=== mortality ===")
for col in df_mortality.columns:
    taux = df_mortality[col].notnull().mean() * 100
    print(f"{col} : {taux:.1f}% rempli")

=== basicsafewater ===
Year : 100.0% rempli
Country : 100.0% rempli
Granularity : 100.0% rempli
Population using at least basic drinking-water services (%) : 89.9% rempli
Population using safely managed drinking-water services (%) : 31.4% rempli

=== mortality ===
Year : 100.0% rempli
Country : 100.0% rempli
Granularity : 100.0% rempli
Mortality rate attributed to exposure to unsafe WASH services : 100.0% rempli
WASH deaths : 33.3% rempli


markdown## Décisions sur les colonnes et le périmètre

### Taux de remplissage des colonnes lacunaires
Le code ci-dessus me confirme les colonnes exploitables :
→ **basic drinking-water services (~90%)** : conservée, indicateur principal d'accès à l'eau  
→ **safely managed drinking-water services (~31%)** : abandonnée, trop peu de données fiables  
→ **Mortality rate (100%)** : conservée  
→ **WASH deaths (~33%)** : renseigné uniquement pour `Granularity = Total`, conservée pour analyses globales uniquement  

### Pas d'imputation
Je choisis de **ne pas imputer** les valeurs manquantes : ce sont des données officielles OMS, toute imputation biaiserait l'analyse. Je travaille sur les données disponibles et reste transparente sur les pays absents.

### Types de données
Les types sont cohérents avec la nature des données (str pour catégoriel, int/float pour numérique). **Aucune conversion nécessaire.**

### Périmètre : pays communs
Les datasets couvrent entre 183 et 239 pays. Pour les analyses croisées, je travaillerai sur l'**intersection des pays communs**, via des jointures sur la clé `Country`. Le volume final sera donc **réduit** — Cela garantit une analyse cohérente et de qualité. 

## 11. Nettoyage et harmonisation

### Objectif
Préparer les datasets pour les jointures : renommer les colonnes, supprimer les colonnes inexploitables, standardiser les noms pour faciliter les croisements.

### Actions prévues
→ **basicsafewater** : drop de la colonne `safely managed drinking-water` (31% rempli), renommage de `basic drinking-water` en nom court  
→ **mortality** : renommage de `Mortality rate attributed to exposure to unsafe WASH services` en nom court  
→ **region_country** : renommage de `REGION (DISPLAY)` → `Region` et `COUNTRY (DISPLAY)` → `Country`  
→ **Vérification** : affichage des colonnes finales pour validation

In [ ]:
# basicsafewater : je drop la colonne trop lacunaire et renomme
df_basicsafewater = df_basicsafewater.drop(columns=['Population using safely managed drinking-water services (%)'])
df_basicsafewater = df_basicsafewater.rename(columns={
    'Population using at least basic drinking-water services (%)': 'basic_water_access_pct'
})

# mortality : je renomme la colonne cible
df_mortality = df_mortality.rename(columns={
    'Mortality rate attributed to exposure to unsafe WASH services': 'mortality_rate'
})

# region_country : j'harmonise les noms de colonnes
df_region_country = df_region_country.rename(columns={
    'REGION (DISPLAY)': 'Region',
    'COUNTRY (DISPLAY)': 'Country'
})



In [12]:
# Je reconstruis le dictionnaire pour qu'il pointe vers les versions à jour
dfs = {
    'population': df_population,
    'basicsafewater': df_basicsafewater,
    'mortality': df_mortality,
    'political': df_political,
    'region_country': df_region_country
}

# Je vérifie à nouveau
for name, df in dfs.items():
    print(f"{name} : {df.columns.tolist()}")

population : ['Country', 'Granularity', 'Year', 'Population']
basicsafewater : ['Year', 'Country', 'Granularity', 'basic_water_access_pct']
mortality : ['Year', 'Country', 'Granularity', 'mortality_rate', 'WASH deaths']
political : ['Country', 'Year', 'Political_Stability', 'Granularity']
region_country : ['Region', 'Country']


## Jointures et création d'un dataset consolidé

### Objectif
Construire un dataset unique qui croise accès à l'eau, mortalité, stabilité politique, population et région. Cela me permettra ensuite de répondre aux questions métier et de construire le score de priorité.

### Stratégie
→ Clé de jointure : **`Country`** (+ `Year` pour les données temporelles)  
→ Type de jointure : **`inner`** → je garde uniquement les pays présents dans tous les datasets  
→ Granularité : je filtre sur **`Total`** pour cette première consolidation (les analyses Rural/Urban et Male/Female se feront à part)  

### Questions à trancher avant de coder
→ **Quelle année de référence ?** La mortalité n'existe que pour 2016. Deux options :  
   - **Option A** : tout aligner sur 2016 (cohérent mais on perd l'info temporelle)  
   - **Option B** : garder plusieurs années et joindre mortalité en broadcast (plus riche mais plus complexe)